# 20 — Director package: maps, star grid, one-pager, deck (spec v1.1, build steps 3–8)

Consumes **19**'s products only (no solves, no recomputation of tiers). Writes
`director_package/figures/*.png`, `director_package/deck_outline.md` and a draft
`director_deck.pptx` (python-pptx; an editable starting point — final polish is Graham/Ethan's).

Every map: locked PAs as a distinct grey layer, graticule with **53°N** emphasized (the E17 tie-in),
scale bar, n-of-formulations note, CVD-safe ramps. Hexes (~250 km², with an ~800 km² variant for
decision c) are PRESENTATION aggregation only — clustering ran at 1 km in 19. Kernel `y2y-geo`.

Decisions still open for Ethan before the final render: (c) hex size, (e) cluster names (placeholders
= nearest named area + bearing), (f) E17 one-pager in the deck vs held for Graham, (h) the
proposed-IPCA dataset (currently the Indigenous-led rows of `y2y_proposed_pa`).


In [ ]:
import importlib, json, pathlib, sys, textwrap
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Patch
from matplotlib.collections import PolyCollection
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize
from matplotlib.cm import ScalarMappable
from scipy import ndimage

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec, director_core as dc
for _m in (config, lc, ec, dc):
    importlib.reload(_m)

ALLOW_PARTIAL = False   # must match 19 (reads its _smoke/ outputs when True)
PKG = dc.PKG / "_smoke" if ALLOW_PARTIAL else dc.PKG
GEO, TAB, FIGD = PKG / "geotiffs", PKG / "tables", PKG / "figures"
assert (PKG / "summary.json").exists(), "run 19_director_surfaces first"
S = json.loads((PKG / "summary.json").read_text())
G = dc.grid()
MAN = pd.read_csv(dc.SPEC / "manifest.csv")

def rd(name):
    with rasterio.open(GEO / name) as src:
        return src.read(1)[G.pu]
Fg, Fp, Ug = rd("F_guarded.tif"), rd("F_unguarded.tif"), rd("union_membership_guarded.tif")
with rasterio.open(GEO / "act_tiers_guarded.tif") as src:
    TIERS = src.read(1)
LAB = dict(np.load(GEO / "cluster_labels.npz"))
PICKS = pd.read_csv(TAB / "picks.csv")
TD1 = pd.read_csv(TAB / "T-D1_cluster_register.csv")
TD2a, TD2b = pd.read_csv(TAB / "T-D2_bands.csv"), pd.read_csv(TAB / "T-D2_acts.csv")
TD3 = pd.read_csv(TAB / "T-D3_scenarios.csv")
E17 = pd.read_csv(TAB / "E17_shifts.csv")
# pooled scenario surfaces, rebuilt from the per-formulation tifs per 19's recorded pooling decisions
POOL = {}
for rec in S["pooling"]:
    sid = rec["scenario"]
    fids = [f for f in MAN[MAN.scenario_id == sid].formulation_id if f in S["forms"]]
    if rec["decision"] == "POOLED":
        POOL[sid] = np.mean([rd(f"f_guarded_{f}.tif") for f in fids], axis=0)
    elif rec["decision"].startswith("SEPARATE"):
        for f in fids:
            POOL[f"{sid}@{'245' if 'ssp245' in f else '585'}"] = rd(f"f_guarded_{f}.tif")
    else:
        POOL[sid] = rd(f"f_guarded_{fids[0]}.tif")
assert set(POOL) == set(S["pool_keys"]), "pooling keys drifted from 19"
CL = {lyr: gpd.read_file(GEO / "clusters.gpkg", layer=lyr) for lyr in gpd.list_layers(GEO / "clusters.gpkg").name}
IP = dc.ipca_layer(G)
HEX = {a: dc.hex_grid(G, a) for a in (dc.HEX_KM2, dc.HEX_KM2_ALT)}

# ---- cartography (pixel space: 1 px = 1 km) ---------------------------------------------------
halo = [pe.withStroke(linewidth=2.5, foreground="white")]
BAND_COLORS = ["#f2f2f2", "#1f3a63", "#5b87ad", "#ffd93b", "#e6550d"]
BAND_BOUNDS = [-0.001, 0.05, 0.30, 0.70, 0.95, 1.001]
BAND_NAMES = ["never (<5% of plans)", "rare (5–30%)", "conditional (30–70%)", "frequent (70–95%)", "always (≥95%)"]
PA_COLOR, IPCA_COLOR, CL_COLOR = "#8f8f8f", "#00897b", "crimson"
FCMAP = plt.get_cmap("viridis").copy(); FCMAP.set_over("#ffd93b")    # ramp to 0.70, hard yellow above
FNORM = Normalize(0, dc.FREQ_THR)
N_NOTE = (f"n = {S['n_formulations']} formulations × 51 near-optimal plans · guarded band "
          f"(no value theme > {100 * S['floor_g']:.0f}% behind)")
PAg = np.full(G.shape, np.nan, np.float32); PAg[G.locked2d] = 1.0

def rings_px(geom):
    polys = geom.geoms if geom.geom_type == "MultiPolygon" else [geom]
    out = []
    for p in polys:
        x, y = np.asarray(p.exterior.coords)[:, :2].T      # some proposals carry a Z coordinate
        px, py = dc.xy_to_px(G, x, y)
        out.append(np.c_[px, py])
    return out

def draw_pa(ax):
    ax.imshow(PAg, cmap=ListedColormap([PA_COLOR]), interpolation="nearest", zorder=1)

def draw_hex(ax, hm, cmap, norm, alpha=1.0):
    ok = hm[np.isfinite(hm.value)]
    verts = [rings_px(g)[0] for g in ok.geometry]
    pc = PolyCollection(verts, facecolors=cmap(norm(ok.value.values)), edgecolors="none", alpha=alpha, zorder=0.5)
    ax.add_collection(pc)

def draw_clusters(ax, layer, numbers, color=CL_COLOR, lw=0.9, fs=10):
    g = CL.get(layer)
    if g is None:
        return
    for _, r in g.iterrows():
        for ring in rings_px(r.geometry):
            ax.plot(ring[:, 0], ring[:, 1], color=color, lw=lw, zorder=4)
        if int(r.cid) in numbers:
            c = r.geometry.centroid
            cx, cy = dc.xy_to_px(G, c.x, c.y)
            ax.annotate(str(numbers[int(r.cid)]), xy=(cx, cy), xytext=(cx + 30, cy - 30), fontsize=fs,
                        fontweight="bold", color=color, path_effects=halo, zorder=6,
                        arrowprops=dict(arrowstyle="-", color=color, lw=0.6))

def draw_ipca(ax, lw=1.0):
    for _, r in IP.gdf.iterrows():
        for ring in rings_px(r.geometry):
            ax.plot(ring[:, 0], ring[:, 1], color=IPCA_COLOR, lw=lw, ls="--", zorder=3.5)

def finish(ax, title, handles, note=N_NOTE, legend_loc="lower left", scale=True):
    dc.graticule(ax, G)
    if scale:
        dc.scalebar(ax, G)
    dc.corner_note(ax, note)
    ax.set_title(title, fontsize=11.5)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    if handles:
        ax.legend(handles=handles, loc=legend_loc, bbox_to_anchor=(-0.02, 0.02), fontsize=8, frameon=True)

def picks_for(act, key=None):
    p = PICKS[PICKS.act == act]
    if key is not None:
        p = p[p.key == key]
    return dict(zip(p.cid.astype(int), p.number.astype(int)))

BASE_HANDLES = [Patch(facecolor=PA_COLOR, label="Existing protected areas (locked in)"),
                Patch(facecolor="none", edgecolor=CL_COLOR, label="numbered = deck picks (top-k by area)")]
def table_png(df, path, title, fs=7.5, scale=1.45, colw=None):
    fig, ax = plt.subplots(figsize=(min(1.6 * len(df.columns) + 2, 22), 0.42 * len(df) + 1.6))
    ax.axis("off")
    t = ax.table(cellText=df.values, colLabels=df.columns, cellLoc="center", loc="center")
    t.auto_set_font_size(False); t.set_fontsize(fs); t.scale(1, scale)
    for j in range(len(df.columns)):
        t[0, j].set_facecolor("#2b4f7d"); t[0, j].set_text_props(color="white", fontweight="bold")
    ax.set_title(title, fontsize=11, pad=12)
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.show()

print(f"loaded package: {len(S['forms'])} formulations | pool keys {list(POOL)} | cluster layers {list(CL)}")


In [ ]:
# ---- slide 2: how to read the map (1 km tiers vs hex 250 vs hex 800, zoom on pick 1) ----------
p1 = PICKS.iloc[0]
lab1 = LAB["act1"]
rows_, cols_ = np.where(lab1 == int(p1.cid))
r0, r1 = max(int(rows_.mean()) - 220, 0), min(int(rows_.mean()) + 220, G.shape[0])
c0, c1 = max(int(cols_.mean()) - 170, 0), min(int(cols_.mean()) + 170, G.shape[1])
fig, axes = plt.subplots(1, 3, figsize=(15, 6.2))
Fg2 = dc.to_grid(G, np.where(G.disc, Fg, np.nan))
axes[0].imshow(Fg2, cmap=ListedColormap(BAND_COLORS), norm=BoundaryNorm(BAND_BOUNDS, 5), interpolation="nearest")
axes[0].set_title("1 km cells: the five reliability tiers\n(what the clustering uses)", fontsize=10)
for ax, area in zip(axes[1:], (dc.HEX_KM2, dc.HEX_KM2_ALT)):
    gdf, hl = HEX[area]
    hm = dc.hex_means(G, gdf, hl, Fg)
    draw_hex(ax, hm, FCMAP, FNORM)
    ax.set_title(f"~{area} km² hexes: mean guarded F\n(presentation only)", fontsize=10)
for ax in axes:
    draw_pa(ax); draw_clusters(ax, "act1", picks_for("Act 1"))
    ax.set_xlim(c0, c1); ax.set_ylim(r1, r0); ax.set_xticks([]); ax.set_yticks([])
handles = [Patch(facecolor=c, label=n) for c, n in zip(BAND_COLORS, BAND_NAMES)] + BASE_HANDLES
axes[0].legend(handles=handles, loc="lower left", fontsize=7, frameon=True)
fig.colorbar(ScalarMappable(norm=FNORM, cmap=FCMAP), ax=axes[1:], shrink=0.7, extend="max", pad=0.01,
             label="mean guarded F per hex (≥0.70 = hard yellow)")
fig.suptitle("How to read the maps: tiers are levels of RELIABILITY across value positions, not a fence", fontsize=12)
fig.savefig(FIGD / "slide2_how_to_read.png", dpi=170, bbox_inches="tight"); plt.show()


In [ ]:
# ---- Act 1: core commitments (hex 250 default, hex 800 variant, 1 km tiers appendix) -----------
ACT1_SENT = ("Act 1 — these areas recur in near-optimal plans no matter whose values prevail,\n"
             "with no value theme left more than 5% behind")
for area, tag in ((dc.HEX_KM2, "hex250"), (dc.HEX_KM2_ALT, "hex800")):
    gdf, hl = HEX[area]
    # Currie-style (a)/(b) pair: (a) frequency alone (PAs read as F = 1, as in every plan); (b) with the
    # locked-PA layer, cluster picks and an inset histogram of guarded F over unprotected land
    hm_all = dc.hex_means(G, gdf, hl, Fg, disc_only=False)
    hm_disc = dc.hex_means(G, gdf, hl, Fg)
    fig, axes = plt.subplots(1, 2, figsize=(15, 11.5))
    ax = axes[0]
    draw_hex(ax, hm_all, FCMAP, FNORM)
    finish(ax, "(a) frequency in near-optimal plans, all land\n(protected areas sit at F = 1: they are in every plan)", None, note="")
    ax = axes[1]
    draw_pa(ax); draw_hex(ax, hm_disc, FCMAP, FNORM); draw_clusters(ax, "act1", picks_for("Act 1"))
    finish(ax, f"(b) with existing protected areas shown; core = F ≥ 0.70 at 1 km\n(mean guarded F per ~{area} km² hex over unprotected land)", BASE_HANDLES)
    ins = ax.inset_axes([0.60, 0.62, 0.36, 0.16])
    ins.hist(Fg[G.disc], bins=40, range=(0, 1), color="#5b87ad")
    ins.axvline(dc.FREQ_THR, color="#b00020", lw=1)
    ins.set_yscale("log"); ins.set_xlabel("guarded F, unprotected cells", fontsize=7); ins.tick_params(labelsize=6)
    ins.set_title(f"core ≥ 0.70: {S['tier_area_pct_disc']['core']:.1f}% of unprotected land", fontsize=7)
    fig.colorbar(ScalarMappable(norm=FNORM, cmap=FCMAP), ax=axes.tolist(), shrink=0.4, extend="max", pad=0.01,
                 label="frequency in near-optimal plans (guarded F; ≥0.70 hard yellow)")
    fig.suptitle(ACT1_SENT, fontsize=12.5, y=0.96)
    fig.savefig(FIGD / f"act1_core_{tag}.png", dpi=180, bbox_inches="tight"); plt.show()
# appendix: the analytic 1 km surface in the five tiers
fig, ax = plt.subplots(figsize=(7.5, 11.5))
ax.imshow(Fg2, cmap=ListedColormap(BAND_COLORS), norm=BoundaryNorm(BAND_BOUNDS, 5), interpolation="nearest")
draw_pa(ax); draw_clusters(ax, "act1", picks_for("Act 1"))
finish(ax, "Appendix — guarded ensemble F at 1 km, five reliability tiers",
       [Patch(facecolor=c, label=n) for c, n in zip(BAND_COLORS, BAND_NAMES)] + BASE_HANDLES)
fig.savefig(FIGD / "act1_core_1km_tiers.png", dpi=200, bbox_inches="tight"); plt.show()
print(f"core tier (guarded F ≥ 0.70): {S['frequent_km2']['guarded'] + S['always_km2']['guarded']:,} km² vs "
      f"unguarded {S['frequent_km2']['unguarded'] + S['always_km2']['unguarded']:,} km²")


In [ ]:
# ---- Act 1 star grid (picks) + the T-D2 doubling slide -------------------------------------------
def star_rows(rows, color):
    return [dict(title=f"{int(r.number)}. {r['name']}\n{r.area_km2:,.0f} km² · mean F {r.mean_guarded_F:.2f}",
                 values={a: r[f"pct_{a}"] for a in dc.STAR_AXES}, color=color) for _, r in rows.iterrows()]
A1 = TD1[TD1.act == "Act 1"].dropna(subset=["number"]).sort_values("number")
dc.plot_star_grid(star_rows(A1, "#2b4f7d"), FIGD / "act1_star_grid.png",
                  "Act 1 core clusters — why these places (block percentiles vs the discretionary landscape)")
plt.show()

fig, ax = plt.subplots(figsize=(11, 4.8))
lab_ = list(TD3.formulation) + ["ENSEMBLE"]
ung = list(TD3.frequent_km2_unguarded) + [S["frequent_km2"]["unguarded"] + S["always_km2"]["unguarded"]]
gua = list(TD3.frequent_km2_guarded) + [S["frequent_km2"]["guarded"] + S["always_km2"]["guarded"]]
x = np.arange(len(lab_))
ax.bar(x - 0.2, ung, 0.4, color="#9ecae1", label="aggregate 5% band (members may sacrifice a theme)")
ax.bar(x + 0.2, gua, 0.4, color="#2b4f7d", label="guarded band (every theme within 5% of its anchor)")
ax.set_xticks(x); ax.set_xticklabels([l.replace("_theta", " θ").replace("_ssp", " ") for l in lab_], rotation=60, ha="right", fontsize=8)
ax.set_ylabel("frequent tier (f ≥ 0.70), km²"); ax.legend(fontsize=8)
ax.set_title("Guardrails roughly double the land that can be PROMISED — the E15 result", fontsize=11)
fig.savefig(FIGD / "td2_doubling.png", dpi=200, bbox_inches="tight"); plt.show()


In [ ]:
# ---- Act 2: value-specific priorities (small multiples, pooled climate levels) -------------------
keys = [k for k in POOL if k.split("@")[0] in dc.ACT2_SCENARIOS]
gdf, hl = HEX[dc.HEX_KM2]
n = len(keys); ncols = min(n, 4) if n else 1
fig, axes = plt.subplots(1, max(ncols, 1), figsize=(4.6 * max(ncols, 1), 9.5), squeeze=False)
for ax, key in zip(axes[0], keys):
    sid = key.split("@")[0]
    hm = dc.hex_means(G, gdf, hl, POOL[key])
    draw_pa(ax); draw_hex(ax, hm, FCMAP, FNORM); draw_ipca(ax)
    draw_clusters(ax, "act1", {}, color="#444444", lw=0.5)
    nums = picks_for("Act 2", key)
    draw_clusters(ax, f"act2_{key}", nums)
    # IPCA alignment call-outs: independent convergence, never assignment
    for _, r in TD1[(TD1.act == "Act 2") & (TD1.driving == key) & TD1.number.notna()].iterrows():
        if r.pct_in_proposed_IPCA >= 5:
            rr = PICKS[PICKS.number == int(r.number)].iloc[0]
            x_, y_ = dc.xy_to_px(G, *__import__("pyproj").Transformer.from_crs("EPSG:4326", G.crs, always_xy=True).transform(rr.lon, rr.lat))
            ax.annotate(f"{r.pct_in_proposed_IPCA:.0f}% inside a declared\nIPCA proposal", xy=(x_, y_), xytext=(x_ - 260, y_ + 120),
                        fontsize=7, color=IPCA_COLOR, path_effects=halo, zorder=6,
                        arrowprops=dict(arrowstyle="->", color=IPCA_COLOR, lw=0.7))
    freq = int(((POOL[key] >= dc.FREQ_THR) & G.disc).sum())
    rec = next(r for r in S["pooling"] if r["scenario"] == sid)
    note = "both climate levels pooled" if rec["decision"] == "POOLED" else ("single level" if rec["levels"] == 1 else f"level {key.split('@')[1]} shown separately")
    finish(ax, f"{dc.SCENARIO_LABEL[sid]}\nfrequent tier {freq:,} km² = {100 * freq / G.n_disc:.1f}% of unprotected land · {note}",
           None, note="", scale=False)
for ax in axes[0][len(keys):]:
    ax.axis("off")
handles = BASE_HANDLES + [Patch(facecolor="none", edgecolor="#444444", label="Act-1 core clusters (already committed)"),
                          Patch(facecolor="none", edgecolor=IPCA_COLOR, linestyle="--", label="declared IPCA proposals (not locked; overlap = convergence)")]
fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=8, frameon=True, bbox_to_anchor=(0.5, 0.03))
fig.colorbar(ScalarMappable(norm=FNORM, cmap=FCMAP), ax=axes[0].tolist(), shrink=0.4, extend="max", pad=0.01,
             label="mean guarded f per hex (≥0.70 hard yellow)")
fig.suptitle("Act 2 — if Y2Y leans into value X, these areas join the core (guarded f per scenario; numbered = picks)\n"
             "ALIGNMENT, not assignment: the analysis sets no priorities inside IPCAs", fontsize=11.5, y=0.995)
fig.text(0.01, 0.005, N_NOTE, fontsize=7.5, color="#333333")
fig.subplots_adjust(bottom=0.1)
fig.savefig(FIGD / "act2_scenarios_hex250.png", dpi=180, bbox_inches="tight"); plt.show()
A2 = TD1[TD1.act == "Act 2"].dropna(subset=["number"]).sort_values("number")
if len(A2):
    dc.plot_star_grid(star_rows(A2, "#a50f2d"), FIGD / "act2_star_grid.png",
                      "Act 2 scenario clusters — block percentiles vs the discretionary landscape")
    plt.show()


In [ ]:
# ---- Act 3: the opportunity landscape (union membership + tier map) + guardrail ----------------
fig, axes = plt.subplots(1, 2, figsize=(15, 11.5))
ax = axes[0]
hm = dc.hex_means(G, gdf, hl, Ug)
draw_pa(ax); draw_hex(ax, hm, plt.get_cmap("cividis"), Normalize(0, 1))
finish(ax, "Share of formulations whose near-optimal band\nincludes the cell in ≥1 plan (mean per hex)", BASE_HANDLES[:1])
fig.colorbar(ScalarMappable(norm=Normalize(0, 1), cmap="cividis"), ax=ax, shrink=0.45, pad=0.01, label="union membership")
ax = axes[1]
T2 = TIERS.astype(float); T2[TIERS == 255] = np.nan
ax.imshow(T2, cmap=ListedColormap(["#f7f7f7", "#c7d9e8", "#5b87ad", "#ffd93b"]), norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], 4),
          interpolation="nearest")
draw_pa(ax)
finish(ax, "Tiers at 1 km: what can be promised about each", 
       [Patch(facecolor="#ffd93b", label="Act 1 core — recurs under every value position"),
        Patch(facecolor="#5b87ad", label="Act 2 — joins the core under a named scenario"),
        Patch(facecolor="#c7d9e8", label="Act 3 opportunity — in ≥1 near-optimal plan"),
        Patch(facecolor="#f7f7f7", label="never selected within 5% of optimal"),
        Patch(facecolor=PA_COLOR, label="Existing protected areas")])
e11 = S["e11_pairs_in_band"]
fig.text(0.5, 0.045, f"Act 3 — the analysis does not forbid working anywhere relationships and feasibility are positive; "
         f"it tells you what can be PROMISED about each tier.\n{S['ever_in_band_pct']['guarded']:.0f}% of unprotected land "
         f"appears in at least one near-optimal plan; {e11[0]} of {e11[1]} ordered pairs of value positions are mutually "
         f"near-optimal (each within 5% of the other's optimum).\nPRIORITY ≠ PERMISSION: tiers are levels of reliability, not a fence.",
         ha="center", fontsize=9.5, color="#222222")
fig.savefig(FIGD / "act3_opportunity.png", dpi=180, bbox_inches="tight"); plt.show()


In [ ]:
# ---- v1.3: tier-achievement figure (zero-solve), anchor agreement matrix, T-D4 render --------------
TA = pd.read_csv(TAB / "tier_achievement.csv"); TA_ref = pd.read_csv(TAB / "tier_achievement_reference.csv", index_col=0)
piv = TA[TA.feature == "BLOCK"].pivot(index="tier", columns="block", values="capture")
tiers_order = ["existing PAs", "+ Act 1 core", "+ Act 2 scenario tiers", "+ Act 3 opportunity"]
piv = piv.reindex(tiers_order)
blocks = ["core_habitat", "connectivity", "biodiversity", "carbon"]
fig, ax = plt.subplots(figsize=(10.5, 5))
x = np.arange(len(blocks)); w = 0.2
tc = ["#8f8f8f", "#ffd93b", "#5b87ad", "#c7d9e8"]
for i, tr in enumerate(tiers_order):
    ax.bar(x + (i - 1.5) * w, piv.loc[tr, blocks].values, w, color=tc[i], label=tr, edgecolor="#333333", lw=0.4)
for j, bl in enumerate(blocks):
    ax.plot([x[j] - 0.42, x[j] + 0.42], [TA_ref.loc[bl, "s0"]] * 2, color="#b00020", lw=1.4)
    ax.fill_between([x[j] - 0.42, x[j] + 0.42], TA_ref.loc[bl, "anchor_min"], TA_ref.loc[bl, "anchor_max"], color="#b00020", alpha=0.12)
ax.set_xticks(x); ax.set_xticklabels([b_.replace("_", " ") for b_ in blocks])
ax.set_ylabel("share of the region's value captured (block mean)"); ax.set_ylim(0, 1)
ax.plot([], [], color="#b00020", lw=1.4, label="balanced anchor (line) · range across all anchors (band)")
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False)
ax.set_title("What each tier delivers: value captured by cumulative tier vs what a single optimal plan captures", fontsize=10.5)
fig.text(0.01, -0.16, "cumulative tiers include existing PAs; the opportunity tier covers most of the region, so its bar ≈ the regional total "
         "— the point is the core + scenario tiers against the anchor band (T-D2 companion, Jung-style achievement view).", fontsize=7.5, color="#444444")
fig.savefig(FIGD / "tier_achievement.png", dpi=200, bbox_inches="tight"); plt.show()

J = pd.read_csv(dc.SPEC / "E11_anchor_jaccard.csv", index_col=0)
fig, ax = plt.subplots(figsize=(8.5, 7.5))
im = ax.imshow(J.values, cmap="cividis", vmin=0.3, vmax=1)
lab_ = [c.replace("_theta", " θ").replace("_ssp", " ") for c in J.columns]
ax.set_xticks(range(len(J))); ax.set_xticklabels(lab_, rotation=60, ha="right", fontsize=7.5)
ax.set_yticks(range(len(J))); ax.set_yticklabels(lab_, fontsize=7.5)
for i in range(len(J)):
    for j in range(len(J)):
        ax.text(j, i, f"{J.values[i, j]:.2f}", ha="center", va="center", fontsize=5.5, color="white" if J.values[i, j] < 0.7 else "black")
fig.colorbar(im, ax=ax, shrink=0.7, label="Jaccard agreement between optimal plans")
ax.set_title("Appendix — agreement between the 14 value positions' optimal plans (anchors; E11 record)", fontsize=10)
fig.savefig(FIGD / "agreement_matrix.png", dpi=200, bbox_inches="tight"); plt.show()

td4 = TAB / "T-D4_ecoregions.csv"
if td4.exists():
    TD4 = pd.read_csv(td4)
    d4 = TD4.copy()
    for c in d4.columns[1:-1]:
        d4[c] = d4[c].map("{:,.0f}".format)
    d4["mean lat"] = d4["mean lat"].map("{:.1f}".format)
    table_png(d4.head(30), FIGD / "td4_ecoregions.png", f"T-D4 — tier area by ecoregion ({S['td4']})")
else:
    print(f"T-D4 render skipped: {S['td4']}")


In [ ]:
# ---- "why these places": driver attribution for the Act-1 picks (E13 in one visual) ---------------
drv = [c for c in TD1.columns if c.startswith("driver_") and "rare-attainable" not in c]
fig, ax = plt.subplots(figsize=(10, 4.6))
x = np.arange(len(A1)); w = 0.8 / len(drv)
for i, c in enumerate(drv):
    ax.bar(x + (i - (len(drv) - 1) / 2) * w, A1[c].values, w, label=c.replace("driver_", ""))
ax.set_xticks(x); ax.set_xticklabels([f"{int(n)}. {nm.split(' (')[0][:30]}" for n, nm in zip(A1.number, A1["name"])], rotation=20, ha="right", fontsize=8)
ax.set_ylabel("% of cluster cells inside the driver mask"); ax.legend(fontsize=8)
ax.set_title("Why these places: core clusters sit on BINDING claims (dense carbon, the connectivity spike, the scarcest ecosystems)\n"
             "— they spike on what is scarce rather than excelling everywhere (E13)", fontsize=10)
fig.text(0.01, -0.16, "disclosed: the spec's rare-ATTAINABLE EFG footprint (36/40 EFGs) covers ~79% of the region and reads "
         "~100% for every cluster, so the scarcest-EFG companion (≤1% footprint each) is shown instead; both in T-D1.",
         fontsize=7.5, color="#444444")
fig.subplots_adjust(bottom=0.28)
fig.savefig(FIGD / "why_these_places.png", dpi=200, bbox_inches="tight"); plt.show()


In [ ]:
# ---- E17 endorsement one-pager ----------------------------------------------------------------------
e = S["e17"]
fig = plt.figure(figsize=(14, 7.6))
ax = fig.add_axes([0.06, 0.14, 0.42, 0.74])
order = ["efg", "biodiversity", "core_habitat", "connectivity", "carbon"]
E = E17.set_index("block_out").reindex(order)
cols = ["#b00020" if b == "efg" else "#5b87ad" for b in order]
ax.barh(range(len(order)), E.delta_lat.values, color=cols)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(["representativeness (40 EFGs) OUT", "biodiversity OUT", "core habitat OUT", "connectivity OUT", "carbon OUT"], fontsize=9)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlim(E.delta_lat.min() - 0.6, E.delta_lat.max() + 0.6)
for i, v in enumerate(E.delta_lat.values):
    ax.text(v + (0.05 if v >= 0 else -0.05), i, f"{v:+.2f}°", va="center", ha="left" if v >= 0 else "right", fontsize=9,
            fontweight="bold" if order[i] == "efg" else "normal")
ax.set_xlabel(f"shift in the plan's mean latitude when one value theme is removed (vs balanced anchor at {e['base_lat']:.1f}°N)")
ax.set_title("Leave-one-theme-out: what pulls the plan south", fontsize=11)
ax.invert_yaxis()
txt = (f"THE FACT  {e['n_efg_south']} of {e['n_efg']} ecosystem functional groups sit >90% south of 53°N "
       f"(median EFG latitude {e['median_efg_lat']:.1f}°N). Removing the representativeness theme moves the balanced plan "
       f"{E.loc['efg', 'delta_lat']:+.2f}° north — the largest single force, larger than any PROACT value theme.\n\n"
       "BOTH TRUTHS  The ~2° southern anchor was never explicitly endorsed by Y2Y — AND it encodes real conservation logic: "
       "rare ecosystems are southern because conversion pressure squeezed them there. Deleting the doctrine would be value "
       "surgery after the results; keeping it silently would be an un-chosen priority.\n\n"
       "THE DECISION  Does Y2Y affirm the representativeness anchor at its measured price (a plan centred ~2° further south "
       "than the four PROACT themes alone would put it)?\n\n"
       "IF NOT  Regionally stratified representation targets (north / south of 53°N) — applied-paper scope; the 40-EFG "
       "adequacy foundation stays, its geography stops being the tie-breaker.\n\n"
       "PRECEDENT  The national 30×30 analysis (Currie, Liang & Snider 2025, WWF-Canada) found the same north–south tension: "
       "species targets pull priorities south into high-footprint ecozones while non-species targets shape the north. "
       "This analysis is the first to decompose each value's pull causally, in degrees of latitude — a known property of "
       "Canadian prioritization, here measured.")
fig.text(0.52, 0.90, "\n".join(textwrap.fill(p, 80) for p in txt.split("\n")), va="top", fontsize=8.6, family="sans-serif")
fig.suptitle("E17 — the representativeness anchor: an un-chosen 2° southern lean, now measured", fontsize=13, y=0.97)
fig.savefig(FIGD / "e17_one_pager.png", dpi=200, bbox_inches="tight"); plt.show()


In [ ]:
# ---- rendered tables for the deck (T-D1 picks, T-D2, T-D3) -------------------------------------------
pk = TD1.dropna(subset=["number"]).sort_values("number")
d1 = pd.DataFrame({
    "#": pk.number.astype(int), "cluster (placeholder name)": pk["name"], "act": pk.act, "driving": pk.driving_label,
    "km²": pk.area_km2.map("{:,.0f}".format), "mean F": pk.mean_guarded_F.map("{:.2f}".format),
    "core hab.": pk["pct_core habitat"].map("{:.2f}".format), "connect.": pk.pct_connectivity.map("{:.2f}".format),
    "biodiv.": pk.pct_biodiversity.map("{:.2f}".format), "carbon": pk.pct_carbon.map("{:.2f}".format),
    "EFG/40": pk.pct_representativeness.map("{:.2f}".format), "intact.†": pk.pct_intactness.map("{:.2f}".format),
    "% θ-tail": pk["driver_m_soc theta-tail"].map("{:.0f}".format),
    "% scarce EFG": pk[[c for c in pk.columns if c.startswith("driver_rarest")][0]].map("{:.0f}".format),
    "lat °N": pk.mean_lat.map("{:.1f}".format), "% in IPCA prop.": pk.pct_in_proposed_IPCA.map("{:.0f}".format),
    "# form. frequent": pk.n_formulations_frequent})
table_png(d1, FIGD / "td1_picks.png", "T-D1 — cluster register (deck picks; full register in tables/T-D1_cluster_register.csv)")
d2 = TD2a.copy()
for c in d2.columns[1:]:
    d2[c] = d2[c].map("{:,.0f}".format if "km2" in c else "{:.1f}".format)
table_png(d2, FIGD / "td2_bands.png", "T-D2 — reliability tiers of the unprotected landscape, unguarded vs guarded semantics")
d2b = TD2b.copy()
for c in d2b.columns[1:]:
    d2b[c] = d2b[c].map(lambda v: "" if pd.isna(v) else ("{:,.0f}".format(v) if isinstance(v, (int, float)) and v > 100 else "{:.1f}".format(v)))
table_png(d2b, FIGD / "td2_acts.png", "T-D2 — act accounting (km² and % of the unprotected landscape)")
TD5 = pd.read_csv(TAB / "T-D5_protected_baseline.csv")
d5 = pd.DataFrame({"value": TD5.value, "% of regional total inside PAs": TD5.pct_of_regional_total_in_PAs.map("{:.1f}".format),
                   "S0 target": TD5.S0_target.map("{:.3f}".format), "% of target already banked": TD5.pct_of_target_already_banked.map("{:.0f}".format),
                   "points still needed from unprotected land": TD5.pct_still_needed_from_unprotected_land.map("{:.1f}".format),
                   "enrichment: existing PAs": TD5.enrichment_existing_PAs.map("{:.2f}×".format),
                   "enrichment: optimizer's new half (S0)": TD5.enrichment_S0_new_half.map("{:.2f}×".format)})
table_png(d5, FIGD / "td5_protected_baseline.png",
          f"T-D5 — what existing protected areas already bank ({S['protected_baseline']['pa_km2']:,} km² = "
          f"{S['protected_baseline']['pa_pct_of_budget']:.0f}% of the 30% budget; {S['protected_baseline']['efg_present_in_PAs']}/40 EFGs present)")
TD5b = pd.read_csv(TAB / "T-D5b_enrichment_by_scenario.csv", index_col=0)
cols = ["existing PAs"] + [c for c in TD5b.columns if c.startswith("anchor") and "585" in c] + \
       [c for c in TD5b.columns if c.startswith("frequent")]
M5 = TD5b[cols].T
M5.index = [i.replace("frequent tier · ", "frequent: ").replace("anchor · ", "anchor: ").replace(" 585", "") for i in M5.index]
M5.columns = [c.replace("irrecoverable_carbon_", "C ").replace("climate_type_", "").replace("aoh_richness_", "").replace("transboundary_", "").replace("human_modification", "intactness") for c in M5.columns]
fig, ax = plt.subplots(figsize=(11, 0.42 * len(M5) + 1.8))
vmax = float(np.nanmax(np.abs(np.log2(M5.values.astype(float)))))
im = ax.imshow(np.log2(M5.values.astype(float)), cmap="RdBu", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xticks(range(len(M5.columns))); ax.set_xticklabels(M5.columns, rotation=30, ha="right", fontsize=8.5)
ax.set_yticks(range(len(M5))); ax.set_yticklabels(M5.index, fontsize=8.5)
for i in range(len(M5)):
    for j in range(len(M5.columns)):
        v = M5.values[i, j]
        ax.text(j, i, "—" if np.isnan(v) else f"{v:.2f}×", ha="center", va="center", fontsize=8,
                color="white" if (not np.isnan(v) and abs(np.log2(v)) > 0.6 * vmax) else "black")
ax.axhline(0.5, color="black", lw=0.8)
n_anchor = sum(1 for i in M5.index if i.startswith("anchor"))
ax.axhline(0.5 + n_anchor, color="black", lw=0.8)
ax.set_title("T-D5b — enrichment (value captured ÷ area used): existing PAs vs each scenario's optimal plan (its new half) "
             "vs what the guarded frequent tiers promise", fontsize=9.5)
fig.colorbar(im, ax=ax, shrink=0.6, label="log2 enrichment (blue < 1× < red)")
fig.savefig(FIGD / "td5b_enrichment_by_scenario.png", dpi=200, bbox_inches="tight"); plt.show()
d3 = pd.DataFrame({"formulation": TD3.formulation, "scenario": TD3.scenario, "climate": TD3.climate,
                   "core hab.": TD3.capture_core_habitat.map("{:.3f}".format), "connect.": TD3.capture_connectivity.map("{:.3f}".format),
                   "carbon": TD3.capture_carbon.map("{:.3f}".format), "biodiv.": TD3.capture_biodiversity.map("{:.3f}".format),
                   "m_soc tail": TD3.tail_m_soc.map("{:.2f}".format), "biomass tail": TD3.tail_biomass.map("{:.2f}".format),
                   "anchor lat": TD3.anchor_mean_lat.map("{:.1f}".format),
                   "frequent km² (guarded)": TD3.frequent_km2_guarded.map("{:,}".format),
                   "D unguard→guard": [f"{a:.3f}→{b:.3f}" for a, b in zip(TD3.D_unguarded, TD3.D_guarded)]})
table_png(d3, FIGD / "td3_scenarios.png", "T-D3 — scenario summary (captures = mean captured fraction of the block's features; tails = θ-tail mass)")


In [ ]:
# ---- deck outline + draft pptx ----------------------------------------------------------------------
e11 = S["e11_pairs_in_band"]
core_km2 = S["frequent_km2"]["guarded"] + S["always_km2"]["guarded"]
slides = [
 dict(title="Where Y2Y's values agree — and where they diverge", image=None,
      bullets=["14 value/climate positions × 51 near-optimal plans each",
               f"30% of the region; existing PAs locked in — they are {S['protected_baseline']['pa_pct_of_region']:.0f}% of the region "
               f"({S['protected_baseline']['pa_pct_of_budget']:.0f}% of the budget) and already bank "
               f"{S['protected_baseline']['banked_min']:.0f}–{S['protected_baseline']['banked_max']:.0f}% of every value "
               f"({S['protected_baseline']['efg_present_in_PAs']}/40 ecosystem groups present) — remarkably AVERAGE land for these values "
               f"(enrichment 0.8–1.3×); the tiers are about the other half, where the optimizer works 1–1.7× harder per km²",
               "Guarded band: no value theme left >5% behind", "Tiers = reliability, not a fence"],
      notes="Context slide. Methods live in the study plan v0.14.1; this package = director_package_spec v1.1."),
 dict(title="How to read the maps", image=FIGD / "slide2_how_to_read.png",
      bullets=["Grey = existing protected areas (fixed in every plan)", "1 km tiers drive the clustering",
               "Hexes (~250 km²) are for legibility only — our DISPLAY hex ≈ the national 30×30 analysis's PLANNING unit (100 km²); "
               "our analysis runs at 1 km², two orders of magnitude finer ('this valley', not 'this ecodistrict')",
               "51 near-optimal plans × 14 value positions, versus 4 plans in the national analysis",
               "53°N marked: see the E17 one-pager"]),
 dict(title="Act 1 — Core commitments", image=FIGD / "act1_core_hex250.png",
      bullets=[f"Core = F ≥ 0.70 across ALL positions: {core_km2:,} km² of unprotected land",
               "These areas recur in near-optimal plans no matter whose values prevail, with no value theme left more than 5% behind",
               "Numbered = the largest core clusters (full register in the appendix)"]),
 dict(title="Why these places + what guardrails buy", image=FIGD / "act1_star_grid.png",
      bullets=["Core clusters spike on BINDING claims (dense soil carbon, scarce ecosystems) rather than excelling everywhere",
               f"Guardrails roughly double the promisable land (see td2_doubling.png): {S['frequent_km2']['unguarded'] + S['always_km2']['unguarded']:,} → {core_km2:,} km²"]),
 dict(title="Act 2 — Value-specific priorities", image=FIGD / "act2_scenarios_hex250.png",
      bullets=["If Y2Y leans into value X, these areas join the core", "Climate levels pooled where they agree (see pooling_check.csv)",
               "Dashed teal = Nations' declared IPCA proposals — overlap is independent convergence, not assignment",
               "Precedent: the national 30×30 analysis (Currie et al. 2025) reports that proposed IPCAs coincide with priority areas and that "
               "Indigenous priorities supersede top-down prioritization — rights and title are not contingent on GBF compatibility"]),
 dict(title="Scenario clusters — value profiles", image=FIGD / "act2_star_grid.png",
      bullets=["Each scenario's two largest clusters outside the core", "Same radial scale as Act 1"]),
 dict(title="Act 3 — The opportunity landscape", image=FIGD / "act3_opportunity.png",
      bullets=[f"{S['ever_in_band_pct']['guarded']:.0f}% of unprotected land appears in ≥1 near-optimal plan",
               f"{e11[0]}/{e11[1]} pairs of value positions are mutually near-optimal",
               "The analysis does not forbid working anywhere relationships and feasibility are positive — it tells you what can be promised about each tier",
               "PRIORITY ≠ PERMISSION"]),
 dict(title="What each tier delivers", image=FIGD / "tier_achievement.png",
      bullets=["Value captured per theme by cumulative tier (PAs → + core → + scenario tiers → + opportunity)",
               "Red line/band = what a single optimal plan captures (balanced; range across all positions)",
               "Pairs with T-D2: the promise per tier, in the currency of each value"]),
 dict(title="Why these places (driver attribution)", image=FIGD / "why_these_places.png",
      bullets=["High frequency follows binding scarcity, not the most-valued layer", "Carbon θ-tail and the scarcest ecosystems pin the core"]),
 dict(title="E17 — an un-chosen 2° southern lean", image=FIGD / "e17_one_pager.png",
      bullets=[f"{S['e17']['n_efg_south']}/{S['e17']['n_efg']} EFGs >90% south of 53°N", "Removing representativeness moves the plan +2.1° north",
               "Decision for Y2Y: affirm the representativeness anchor at its measured price?"]),
 dict(title="What we can promise", image=FIGD / "td2_bands.png",
      bullets=["Core: recurs under every value position (commit)", "Scenario tiers: join the core if that value leads (choose)",
               "Opportunity: defensible wherever feasibility is positive (enable)", "Next: cluster naming, hex size, IPCA dataset confirmation"]),
]
lines = [f"# Y2Y director package — deck outline (spec v1.1)\n", f"_generated from {PKG.relative_to(ROOT)}; n = {S['n_formulations']} formulations_\n"]
for i, s in enumerate(slides, 1):
    lines.append(f"## Slide {i} — {s['title']}\n")
    if s.get("image"):
        lines.append(f"figure: `{pathlib.Path(s['image']).name}`\n")
    lines += [f"- {b}" for b in s.get("bullets", [])]
    if s.get("notes"):
        lines.append(f"\n> {s['notes']}")
    lines.append("")
lines += ["## Appendix", "- `act1_core_hex800.png` (decision c variant), `act1_core_1km_tiers.png` (analytic surface)",
          "- `td1_picks.png` + `tables/T-D1_cluster_register.csv` (full register), `tables/cluster_sensitivity.csv` (0.60/0.80)",
          "- `td2_acts.png`, `td3_scenarios.png`, `td5_protected_baseline.png` (what PAs already bank), `td5b_enrichment_by_scenario.png`, `tables/pooling_check.csv`, 1 km GeoTIFFs in `geotiffs/`",
          "- `agreement_matrix.png` (pairwise Jaccard between the 14 optimal plans), `td4_ecoregions.png` + `tables/T-D4_ecoregions.csv` "
          "(tier area by ecoregion — needs an ecoregion layer in `input_data/ecoregions/`)",
          "- Open decisions: (c) hex size, (e) cluster names, (f) E17 placement, (h) proposed-IPCA dataset"]
(PKG / "deck_outline.md").write_text("\n".join(lines))
dc.build_deck(slides, PKG / "director_deck.pptx")
print(f"wrote {PKG.relative_to(ROOT)}/deck_outline.md + director_deck.pptx ({len(slides)} slides)")
